In [ ]:
import filecmp
import os.path

# Inspired by https://stackoverflow.com/questions/4187564/recursively-compare-two-directories-to-ensure-they-have-the-same-files-and-subdi
def equal_dirs(dir1, dir2, print_identical, print_non_identical):
    dirs_cmp = filecmp.dircmp(dir1, dir2)
    if not compare_dir_files(dir1, dir2, print_identical, print_non_identical):
        pass
        # return False # If you want to exit immediately
    for common_dir in dirs_cmp.common_dirs:
        temp_dir1 = os.path.join(dir1, common_dir)
        temp_dir2 = os.path.join(dir2, common_dir)
        if not equal_dirs(temp_dir1, temp_dir2, print_identical, print_non_identical):
            return False
    return True

def compare_dir_files(path_dir1, path_dir2, print_identical, print_non_identical):
    equal = True
    files_dir1: set[str] = set(os.listdir(path_dir1))
    files_dir2: set[str] = set(os.listdir(path_dir2))
    files_not_shared: list[str] = list(files_dir1 ^ files_dir2)
    if files_not_shared:
        print("Files found which are not shared:")
        for f in files_not_shared:
            print(f, "\n")
        print("")
        
    # For all json files compare their CONTENTS, not just their metadata.
    files_shared: list[str] = list(files_dir1 & files_dir2)
    files_shared = [f for f in files_shared if f.endswith('.json')]
    if files_shared:
        for filename in files_shared:
            identical = filecmp.cmp(os.path.join(path_dir1, filename), os.path.join(path_dir2, filename), shallow=False)
            if not identical and print_non_identical:
                print(f"FAIL: File {filename} is not identical.")
                print(f"Dir1: {path_dir1}")
                print(f"Dir2: {path_dir2}")
                print()
                equal = False
            elif identical and print_identical:
                print(f"OK: File {filename} is identical.")
                print(f"Dir1: {path_dir1}")
                print(f"Dir2: {path_dir2}")
                print()

    return equal

show_matches = False
show_mismatches = True

# Check TraceVerifier simulation query results for similarity. We use run_0, because execution.log will not be the same due to the timestamps.
path_a = f"../dataset/benchmark_raw/run_0"
path_b = f"../dataset/benchmark_raw_rerun/run_0"
equal = equal_dirs(path_a, path_b, show_matches, show_mismatches)
print("Simulation query results identical: ", equal)

In [ ]:
# Check frag verif output for similarity
path_a = f"../dataset/fragverif_raw/run_0"
path_b = f"../dataset/fragverif_raw_rerun/run_0"
equal = equal_dirs(path_a, path_b, show_matches, show_mismatches)
print("Point conversion: ", equal)

In [ ]:
# Check synthetic traces output for similarity.
path_traces_a = "../dataset/allocation_traces_256KB/synthetic"
path_traces_b = "../dataset/allocation_traces_256KB_rerun/synthetic"
equal = equal_dirs(path_traces_a, path_traces_b, show_matches, show_mismatches)
print("Synthetic trace generation identical: ", equal)

In [ ]:
# Check program trace output for similarity
path_traces_a = "../dataset/allocation_traces_256KB/program"
path_traces_b = "../dataset/allocation_traces_256KB_rerun/program"
equal = equal_dirs(path_traces_a, path_traces_b, show_matches, show_mismatches)
print("Program trace conversion identical: ", equal)